In [1]:
from games.nocca_nocca.nocca_nocca import NoccaNocca
from games.nocca_nocca.nocca_nocca_eval import NoccaNoccaEval
from agents.agent_random import RandomAgent
from agents.minimax import MiniMax
from agents.minimax_alphabeta import MiniMaxAlphaBeta
from agents.mcts_t import MonteCarloTreeSearch
from tqdm import tqdm

In [2]:
game = NoccaNocca(max_steps=150, initial_player=0, seed=1)

In [3]:
agents = {
    game.agents[0]: MiniMaxAlphaBeta(game=game, agent=game.agents[0], depth=2, alphabeta=True),
    game.agents[1]: MiniMaxAlphaBeta(game=game, agent=game.agents[1], depth=2, alphabeta=True),
}

In [4]:
game.reset()
player = agents[game.agent_selection]
action = player.action()
print(f"Action taken by {player.agent} (Depth={player.depth}): {action} in {player.iterations} iterations")

Action taken by Black (Depth=2): 62 in 63 iterations


In [5]:
game.reset()
print(f"Initial Agent: {game.agent_selection}")
while not game.game_over():
    game.render()
    action = agents[game.agent_selection].action()
    print(f"Turn {game.steps} -- Agent {game.agent_selection} plays action {action}")
    game.step(action=action)
game.render()
if game.truncated():
    print("Game was truncated")
for agent in agents:
    print(f"Reward agent {agent}: {game.reward(agent)}")
print(f"The winner is: {game.check_for_winner()}")

Initial Agent: Black
0: ___ ___ ___ ___ ___ 
1: 0__ 0__ 0__ 0__ 0__ 
2: ___ ___ ___ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: 1__ 1__ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 0 -- Agent Black plays action 57
0: ___ ___ ___ ___ ___ 
1: 0__ 0__ ___ 0__ 0__ 
2: ___ ___ 0__ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: 1__ 1__ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 1 -- Agent White plays action 243
0: ___ ___ ___ ___ ___ 
1: 0__ 0__ ___ 0__ 0__ 
2: ___ ___ 0__ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: ___ 11_ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 2 -- Agent Black plays action 70
0: ___ ___ ___ ___ ___ 
1: 0__ 0__ ___ ___ 0__ 
2: ___ ___ 0__ ___ 0__ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: ___ 11_ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 3 -- Agent White plays action 266
0: ___ ___ ___ ___ ___ 
1: 0__ 0__ ___ ___ 0__ 
2: ___ __

Para explorar el desempeño de un agente minimax sobre este ambiente es necesario implementar una función de evaluación (NoccaNocca no expone una predefinida como lo hace Ta-Te-Ti).

Implementamos una función de evaluación custom en NoccaNoccaEval con la siguiente heurística:

$V(s) = alpha * D(s)$

Donde:

$D(s) = distance_p(s) - distance_{1-p}(s)$

$distance_p$ es la distancia entre el final del tablero y la ficha más cercana no bloqueada de $p$. No bloqueada quiere decir que no tiene ninguna ficha del oponente por encima.

In [6]:
game = NoccaNoccaEval(max_steps=150, initial_player=0, seed=1, alpha=-1)

agents = {
    game.agents[0]: MiniMaxAlphaBeta(game=game, agent=game.agents[0], depth=1, alphabeta=True),
    game.agents[1]: MiniMaxAlphaBeta(game=game, agent=game.agents[1], depth=4, alphabeta=True),
}

game.reset()
while not game.game_over():
    game.render()
    agent = agents[game.agent_selection]
    val = game.eval(agent.agent)
    action = agent.action()

    print(f"Turn {game.steps} - Distance for agent {game.agent_selection}: {game.distance_to_terminal(agent.agent)} - Eval: {val} - Action {action}\n")
    
    game.step(action=action)

game.render()


0: ___ ___ ___ ___ ___ 
1: 0__ 0__ 0__ 0__ 0__ 
2: ___ ___ ___ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: 1__ 1__ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 0 - Distance for agent Black: 6 - Eval: 0 - Action 68

0: ___ ___ ___ ___ ___ 
1: 0__ 0__ 0__ ___ 0__ 
2: ___ ___ 0__ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: 1__ 1__ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 1 - Distance for agent White: 6 - Eval: -1 - Action 243

0: ___ ___ ___ ___ ___ 
1: 0__ 0__ 0__ ___ 0__ 
2: ___ ___ 0__ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: ___ 11_ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 2 - Distance for agent Black: 5 - Eval: 1 - Action 97

0: ___ ___ ___ ___ ___ 
1: 0__ 0__ 0__ ___ 0__ 
2: ___ ___ ___ ___ ___ 
3: ___ ___ 0__ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: ___ 11_ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 3 - Distance for agent White: 6 - Eval: -2 - 

Agregando esta simple función de evaluación se puede apreciar un comportamiento mucho más inteligente de los agentes que implementan minimax. Se puede ver como ahora los agentes tienden a avanzar sus fichas hacia el lado contrario del tablero, e incluso tienden a bloquear las fichas del otro agente con el objetivo de minimizar la distancia del rival a la meta.

En reiteradas iteraciones del juego se observa que una vez bloqueada la pieza de un rival que se encuentra próxima a ganar, el bloqueo se mantiene indefinidamente, no permitiendo al rival avanzar y desplegando otras piezas para buscar la victoria.

También se aprecia que los agentes con mayor profundidad `depth` tienden a ganar por sobre los agentes con menor profunidad. Se pudo observar que el aumento en la profundidad de exploración permite a los agentes encontrar caminos que se alejan de las fichas del rival cuando se encuentra cerca de la meta (escapando de un bloqueo) mientras que persiguen y bloquean a las fichas del rival cuando estas se encuentran cerca de su objetivo (buscando bloquearlas).

Sin embargo, la demora en la toma de acciones crece exponencialmente a medida que se aumenta el valor de `depth`, imposibilitando la experimentación con valores mayores a 5.

Probaremos ahora el desempeño de MCTS para este tipo de juegos.

In [7]:
game = NoccaNoccaEval(max_steps=150, initial_player=0, seed=1, alpha=-1)

agents = {
    game.agents[0]: MonteCarloTreeSearch(game=game, agent=game.agents[0], simulations=100, rollouts=5, rollout_iterations=50),
    game.agents[1]: RandomAgent(game=game, agent=game.agents[1]),
}

game.reset()
player = agents[game.agent_selection]
action = player.action()
print(f"Action taken by {player.agent} (Rollouts={player.rollouts}): {action} in {player.simulations} simulations")

Action taken by Black (Rollouts=5): 62 in 100 simulations


In [8]:
game.reset()
while not game.game_over():
    game.render()
    agent = agents[game.agent_selection]
    val = game.eval(agent.agent)
    action = agent.action()

    print(f"Turn {game.steps} - Eval: {val} - Action {action}\n")
    
    game.step(action=action)

game.render()

0: ___ ___ ___ ___ ___ 
1: 0__ 0__ 0__ 0__ 0__ 
2: ___ ___ ___ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: 1__ 1__ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 0 - Eval: 0 - Action 62

0: ___ ___ ___ ___ ___ 
1: 0__ 0__ ___ 0__ 0__ 
2: ___ ___ ___ 0__ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: 1__ 1__ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 1 - Eval: -1 - Action 261

0: ___ ___ ___ ___ ___ 
1: 0__ 0__ ___ 0__ 0__ 
2: ___ ___ ___ 0__ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ 1__ ___ ___ ___ 
6: 1__ 1__ ___ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 2 - Eval: 0 - Action 110

0: ___ ___ ___ ___ ___ 
1: 0__ 0__ ___ 0__ 0__ 
2: ___ ___ ___ ___ ___ 
3: ___ ___ ___ ___ 0__ 
4: ___ ___ ___ ___ ___ 
5: ___ 1__ ___ ___ ___ 
6: 1__ 1__ ___ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 3 - Eval: -1 - Action 212

0: ___ ___ ___ ___ ___ 
1: 0__ 0__ ___ 0__ 0__ 
2: ___ ___ ___ ___ ___ 
3: ___ ___ ___ ___ 0__ 
4: ___ ___ 